In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

In [3]:
# -------------------------
# Settings
# -------------------------
os.makedirs('outputs', exist_ok=True)
sns.set_style('whitegrid')  # seaborn style for nicer plots
plt.rcParams.update({'figure.autolayout': True})

# Utility to save fig reliably
def save_fig(fig, filename, dpi=200):
    path = os.path.join('outputs', filename)
    fig.savefig(path, dpi=dpi)
    plt.close(fig)
    print(f"Saved: {path}")

In [4]:
# -------------------------
# Load data (json or csv)
# -------------------------
def load_data():
    if os.path.exists('fire_deaths.json'):
        df = pd.read_json('fire_deaths.json')
    elif os.path.exists('fire_deaths.csv'):
        df = pd.read_csv('fire_deaths.csv', encoding='utf-8')
    else:
        raise FileNotFoundError("Place 'fire_deaths.json' or 'fire_deaths.csv' in the working folder.")
    return df

df = load_data()

In [5]:
# Normalize column names (strip)
df.columns = [c.strip() for c in df.columns]

# If there is an unnamed empty column, rename to Extra
if '' in df.columns:
    df = df.rename(columns={'': 'Extra'})

In [6]:
# -------------------------
# Clean & derived columns
# -------------------------
# Age numeric
df['Age'] = pd.to_numeric(df.get('Age'), errors='coerce')

# Parse dates (try flexible parsing)
df['IncidentDate'] = pd.to_datetime(df.get('Date of Incident'), errors='coerce', dayfirst=False)
df['DeathDate'] = pd.to_datetime(df.get('Date of Death'), errors='coerce', dayfirst=False)

In [7]:
# Derived fields
df['IncidentYear'] = df['IncidentDate'].dt.year
df['IncidentMonth'] = df['IncidentDate'].dt.month
df['IncidentMonthName'] = df['IncidentDate'].dt.month_name()
df['DaysToDeath'] = (df['DeathDate'] - df['IncidentDate']).dt.days

In [8]:
# Age groups
def age_group(a):
    if pd.isna(a): return 'Unknown'
    a = int(a)
    if a < 30: return '<30'
    if a < 40: return '30-39'
    if a < 50: return '40-49'
    if a < 60: return '50-59'
    if a < 70: return '60-69'
    return '70+'

df['AgeGroup'] = df['Age'].apply(age_group)

In [9]:
# Fill NaNs for categorical plotting
for col in ['Cause Of Death','Nature Of Death','Duty','Activity','Emergency','Property Type','Classification','Rank']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

In [10]:
# Save cleaned CSV for Power BI or later use
clean_path = os.path.join('outputs','fire_deaths_clean.csv')
df.to_csv(clean_path, index=False)
print(f"Cleaned data saved to {clean_path}")

Cleaned data saved to outputs\fire_deaths_clean.csv


In [11]:
# -------------------------
# Chart 1: Fatalities by Year + Rolling 5-year average
# -------------------------
by_year = df.groupby('IncidentYear').size().reset_index(name='Deaths').dropna(subset=['IncidentYear']).sort_values('IncidentYear')
by_year['Rolling5yrAvg'] = by_year['Deaths'].rolling(window=5, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(by_year['IncidentYear'], by_year['Deaths'], marker='o', label='Deaths')
ax.plot(by_year['IncidentYear'], by_year['Rolling5yrAvg'], marker='o', linestyle='--', label='Rolling 5yr Avg')
ax.set_title('Fatalities by Incident Year (with 5-year rolling avg)')
ax.set_xlabel('Year')
ax.set_ylabel('Deaths')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend()
save_fig(fig, 'deaths_by_year_with_rolling5.png')

Saved: outputs\deaths_by_year_with_rolling5.png


In [12]:
# -------------------------
# Chart 2: Top Causes of Death (horizontal bar)
# -------------------------
top_causes = df['Cause Of Death'].value_counts().reset_index()
top_causes.columns = ['Cause','Count']

fig, ax = plt.subplots(figsize=(10,6))
sns.barplot(data=top_causes.head(15), y='Cause', x='Count', ax=ax)
ax.set_title('Top Causes of Death (Top 15)')
ax.set_xlabel('Count')
ax.set_ylabel('')
save_fig(fig, 'top_causes_top15.png')

Saved: outputs\top_causes_top15.png


In [13]:
# -------------------------
# Chart 3: Nature of Death by Duty (stacked bar)
# -------------------------
if 'Duty' in df.columns and 'Nature Of Death' in df.columns:
    pivot_dn = pd.crosstab(df['Duty'], df['Nature Of Death'])
    # Keep top duties for visibility
    top_duties = df['Duty'].value_counts().head(8).index
    pivot_small = pivot_dn.loc[pivot_dn.index.intersection(top_duties)]
    fig = plt.figure(figsize=(12,6))
    ax = pivot_small.plot(kind='bar', stacked=True, figsize=(12,6))
    ax.set_title('Nature Of Death by Duty (Top Duties)')
    ax.set_xlabel('Duty')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    save_fig(plt.gcf(), 'nature_of_death_by_duty_stacked.png')
else:
    print("Skipping nature-by-duty plot - missing columns.")

Saved: outputs\nature_of_death_by_duty_stacked.png


<Figure size 1200x600 with 0 Axes>

In [14]:
# -------------------------
# Chart 4: Activity counts (top activities)
# -------------------------
if 'Activity' in df.columns:
    activity_counts = df['Activity'].value_counts().reset_index()
    activity_counts.columns = ['Activity','Count']
    fig, ax = plt.subplots(figsize=(10,6))
    sns.barplot(data=activity_counts.head(20), x='Count', y='Activity', ax=ax)
    ax.set_title('Top Activities associated with fatalities (Top 20)')
    save_fig(fig, 'activity_counts_top20.png')

Saved: outputs\activity_counts_top20.png


In [15]:
# -------------------------
# Chart 5: Age group distribution
# -------------------------
age_dist = df['AgeGroup'].value_counts().reindex(['<30','30-39','40-49','50-59','60-69','70+','Unknown']).fillna(0).reset_index()
age_dist.columns = ['AgeGroup','Count']
fig, ax = plt.subplots(figsize=(8,5))
sns.barplot(data=age_dist, x='AgeGroup', y='Count', ax=ax)
ax.set_title('Age Group Distribution')
save_fig(fig, 'age_group_distribution.png')

Saved: outputs\age_group_distribution.png


In [16]:
# -------------------------
# Chart 6: Emergency vs Non-emergency (pie)
# -------------------------
if 'Emergency' in df.columns:
    emergency_counts = df['Emergency'].value_counts().reset_index()
    emergency_counts.columns = ['Emergency','Count']
    fig, ax = plt.subplots(figsize=(6,6))
    ax.pie(emergency_counts['Count'], labels=emergency_counts['Emergency'], autopct='%1.1f%%', startangle=90)
    ax.set_title('Emergency vs Non-Emergency Fatalities')
    save_fig(fig, 'emergency_vs_non_emergency_pie.png')

Saved: outputs\emergency_vs_non_emergency_pie.png


In [17]:
# -------------------------
# Chart 7: Top risky combos (Duty + Activity + Property Type) -> table & heatmap
# -------------------------
if {'Duty','Activity','Property Type'}.issubset(df.columns):
    combos = df.groupby(['Duty','Activity','Property Type']).size().reset_index(name='Deaths').sort_values('Deaths',ascending=False).head(100)
    combos.to_csv(os.path.join('outputs','top_duty_activity_property_combos.csv'), index=False)

    # Create a condensed pivot for a heatmap — choose top Duty & top Activity
    top_duty = df['Duty'].value_counts().head(8).index
    top_activity = df['Activity'].value_counts().head(12).index
    pivot_heat = pd.crosstab(df.loc[df['Duty'].isin(top_duty) & df['Activity'].isin(top_activity), 'Duty'],
                             df.loc[df['Duty'].isin(top_duty) & df['Activity'].isin(top_activity), 'Activity'])
    # ensure numeric
    pivot_heat = pivot_heat.fillna(0)
    fig, ax = plt.subplots(figsize=(12,6))
    sns.heatmap(pivot_heat, annot=True, fmt='g', ax=ax)
    ax.set_title('Heatmap: Duty vs Activity (counts) — top duties & activities')
    save_fig(fig, 'duty_activity_heatmap_top.png')
else:
    print("Skipping combos heatmap - missing one of Duty/Activity/Property Type")

Saved: outputs\duty_activity_heatmap_top.png


In [18]:
# -------------------------
# Chart 8: Monthly seasonality (aggregate across years)
# -------------------------
if 'IncidentMonth' in df.columns:
    month_order = list(range(1,13))
    month_counts = df.groupby('IncidentMonth').size().reindex(month_order, fill_value=0).reset_index(name='Deaths')
    month_counts['MonthName'] = month_counts['IncidentMonth'].apply(lambda m: pd.to_datetime(str(m), format='%m').month_name())
    fig, ax = plt.subplots(figsize=(10,4))
    sns.lineplot(data=month_counts, x='MonthName', y='Deaths', marker='o', ax=ax)
    ax.set_title('Seasonality: Fatalities by Month (aggregate across years)')
    ax.set_xlabel('')
    save_fig(fig, 'monthly_seasonality.png')

Saved: outputs\monthly_seasonality.png


In [19]:
# -------------------------
# Chart 9: Same-day vs Delayed deaths
# -------------------------
same_day_count = ((df['DaysToDeath'] == 0).sum())
delayed_count = ((df['DaysToDeath'] > 0).sum())
unknown_count = df['DaysToDeath'].isna().sum()
sd_df = pd.DataFrame({
    'Type': ['Same Day','Delayed (>0 days)','Unknown/No Date'],
    'Count': [same_day_count, delayed_count, unknown_count]
})
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=sd_df, x='Type', y='Count', ax=ax)
ax.set_title('Same-day vs Delayed Deaths')
ax.set_xlabel('')
save_fig(fig, 'same_day_vs_delayed.png')

Saved: outputs\same_day_vs_delayed.png


In [20]:
# -------------------------
# Chart 10: Average age by Cause of Death (top causes)
# -------------------------
if 'Cause Of Death' in df.columns:
    avg_age_by_cause = df.groupby('Cause Of Death')['Age'].mean().dropna().sort_values(ascending=False).reset_index()
    fig, ax = plt.subplots(figsize=(10,6))
    sns.barplot(data=avg_age_by_cause.head(20), x='Age', y='Cause Of Death', ax=ax)
    ax.set_title('Average age by Cause of Death (Top 20 by avg age)')
    save_fig(fig, 'avg_age_by_cause_top20.png')

Saved: outputs\avg_age_by_cause_top20.png


In [21]:
# -------------------------
# Save some summary CSVs for Power BI import 
# -------------------------
by_year.to_csv(os.path.join('outputs','deaths_by_year.csv'), index=False)
top_causes.to_csv(os.path.join('outputs','top_causes.csv'), index=False)
age_dist.to_csv(os.path.join('outputs','age_distribution.csv'), index=False)
sd_df.to_csv(os.path.join('outputs','same_day_delayed_summary.csv'), index=False)
if 'Emergency' in df.columns:
    emergency_counts.to_csv(os.path.join('outputs','emergency_counts.csv'), index=False)